# Label Powerset - Starter Notebook
This notebook converts each unique label combination into a single multiclass target and trains a label powerset classifier.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_multilabel_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

In [ ]:
def to_labelset(y_row):
    labels = np.where(y_row == 1)[0]
    return '|'.join(map(str, labels.tolist())) if len(labels) else 'none'


def from_labelset(token, n_classes):
    vec = np.zeros(n_classes, dtype=int)
    if token != 'none':
        for idx in token.split('|'):
            vec[int(idx)] = 1
    return vec


X, Y = make_multilabel_classification(n_samples=950, n_features=18, n_classes=6, n_labels=2, random_state=42)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=42)

train_tokens = np.array([to_labelset(row) for row in Y_train])
test_tokens = np.array([to_labelset(row) for row in Y_test])

In [ ]:
lp_model = LogisticRegression(max_iter=1200)
lp_model.fit(X_train, train_tokens)
pred_tokens = lp_model.predict(X_test)

Y_pred = np.vstack([from_labelset(token, Y.shape[1]) for token in pred_tokens])

results = pd.DataFrame([
    {'Metric': 'Micro F1', 'Value': f1_score(Y_test, Y_pred, average='micro')},
    {'Metric': 'Macro F1', 'Value': f1_score(Y_test, Y_pred, average='macro')},
    {'Metric': 'Exact Match Ratio', 'Value': np.mean((Y_test == Y_pred).all(axis=1))},
]).set_index('Metric').round(3)
results